# Accessible Technical Concept Explainer — Audio Upgrade

A conversational tool that takes a technical question or concept and explains it in a way that is optimized for screen reader and keyboard-only users — people who cannot rely on diagrams, visual metaphors, or "as you can see" phrasing to understand technical ideas. 
The tool is built twice: once using the Anthropic API for nuanced, context-aware explanations, and once using a local model, Llama 3.2, to demonstrate the same concept running entirely offline without an API key. 
The explanation format is deliberately structured for linear audio consumption: a one-sentence plain language summary first, followed by an analogy that uses no visual references, followed by a step-by-step breakdown using numbered points that a screen reader announces clearly, followed by a concrete real-world example grounded in something a blind or low vision user would encounter in daily life, and ending with one follow-up question the user might want to ask next. 
The tool avoids spatial metaphors ("as shown above," "the box on the left," "the diagram below"), color references used as the sole means of conveying information, and ASCII art or table-based layouts that become noise when read aloud.

## Audio Upgrade

This version adds text-to-speech audio output as the primary delivery mechanism rather than an afterthought. The entire purpose of this tool is to make technical explanations work without visual output, so making text-to-speech the default closes the feedback loop the tool was designed for: an explanation that sounds good when spoken is fundamentally different from one that merely avoids visual metaphors on the page, and the only way to verify that difference is to hear it.

The developer asks a technical question, the tool generates an explanation structured for linear audio consumption, and then speaks it aloud using OpenAI's TTS API so the user can evaluate whether the phrasing, pacing, and structure actually work when heard rather than read.

The system prompt and user prompt template have been updated to instruct the model to write in plain prose without markdown formatting characters, since asterisks, hashes, and backticks are read aloud as literal characters by TTS engines.

Both Claude and Ollama responses are shown in the chat for comparison. Claude's response is selected for audio playback to keep the UI to a single audio widget.

Audio output requires an `OPENAI_API_KEY` in your `.env` file. The text-only `run_explainer()` function remains available and requires only the `ANTHROPIC_API_KEY`.

In [ ]:
# imports

import os
import io
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Audio, display

In [ ]:
# Load environment variables

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")


In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

anthropic_url = "https://api.anthropic.com/v1/"
ollama_url = "http://localhost:11434/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [ ]:
system_prompt = """
You are a technical educator specializing in explaining complex concepts 
to blind and low vision learners who use screen readers and keyboard-only 
navigation. Your explanations are optimized for audio consumption — 
they will be read aloud by a text-to-speech engine, not read visually.

Follow these rules on every response:

STRUCTURE — always use this exact order:
1. One-sentence plain language summary of the concept
2. An analogy that uses no visual references — 
   base it on sound, touch, movement, sequence, or 
   everyday non-visual experience
3. Step-by-step breakdown — use numbered steps, 
   each step one sentence, so the listener can 
   follow them as a clear sequence
4. A concrete example grounded in something a blind or 
   low vision user encounters in daily life — 
   VoiceOver navigation, Braille display output, 
   keyboard shortcuts, accessible app behavior
5. One natural follow-up question the learner might want 
   to explore next

INTRODUCE each section by speaking its name in a sentence:
"Here is the summary.", "Here is the analogy.", 
"Here are the steps.", "Here is a concrete example.",
"Here is a follow-up question you might explore next."
This signals structure for listeners without relying on 
visual formatting.

LANGUAGE RULES — never use:
- Visual spatial metaphors: "as shown above", "the box on the left", 
  "the diagram below", "as you can see", "looking at this"
- Color as the sole means of conveying meaning: 
  instead of "the red items are errors" say 
  "items marked as errors"
- ASCII art, tables, or visual layouts that 
  become noise when read aloud — use numbered 
  or bulleted lists instead
- Parenthetical asides that interrupt the listening flow — 
  place all context in the main sentence

TTS FORMATTING — your response will be converted to speech immediately:
- Use plain prose only. Do not use markdown formatting characters — 
  no asterisks, underscores, hashes, backticks, or horizontal rules — 
  because text-to-speech engines speak them as literal characters.
- Spell out all operators, symbols, and punctuation when they are 
  the subject of explanation: say "double question mark operator" 
  not "??" operator. Say "bang equals" not "!=" operator.
- Embed short code tokens inline with surrounding prose rather than 
  isolating them in code blocks: say "the speechRate variable" 
  rather than placing speechRate on its own line in backticks.
- Write numbers out as words in prose context: 
  "four point five to one" not "4.5:1."

TONE — practitioner to practitioner. 
You are not simplifying for a beginner unless asked. 
You are removing visual assumptions from a technical explanation, 
not dumbing it down. The learner is technically capable — 
they just need an explanation that works without a screen.
"""


In [ ]:
user_prompt_template = """
Please explain the following technical concept in a way that is 
fully accessible to a screen reader user. 
Follow your structured explanation format.

Concept or question: {user_question}

If this concept is typically explained using a diagram, 
flowchart, or visual metaphor, explicitly acknowledge that 
and provide an alternative explanation grounded in 
sequence, process, or analogy instead.

Your response will be converted to text-to-speech audio immediately 
after it is generated. Write in plain prose with no markdown 
formatting characters. Introduce each section by speaking its name 
in a full sentence rather than using a visual header or bold label.
"""


In [ ]:
def explain_with_anthropic(question):
    """Explain a technical concept using the Anthropic API."""
    response = anthropic.chat.completions.create(
        model="claude-sonnet-4-6",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content":
                user_prompt_template.format(user_question=question)}
        ]
    )
    return response.choices[0].message.content

In [ ]:
def explain_with_ollama(question, model="llama3.2"):
    """
    Explain a technical concept using a local Ollama model.
    Runs entirely offline — no API key required.
    """
    response = ollama.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content":
                user_prompt_template.format(user_question=question)}
        ]
    )
    return response.choices[0].message.content

In [ ]:
def run_explainer():
    """
    Main loop — runs both models on the same question
    so you can compare output quality.
    """
    print("=== Accessible Technical Concept Explainer ===")
    print("Explanations are structured for screen reader consumption.")
    print("Type 'quit' to exit.\n")

    question = input("Enter your technical question: ").strip()

    print("\n--- Anthropic (Claude) response ---\n")
    try:
        anthropic_response = explain_with_anthropic(question)
        print(anthropic_response)
    except Exception as e:
        print(f"Anthropic call failed: {e}")

    print("\n--- Ollama response (local, offline) ---\n")
    try:
        ollama_response = explain_with_ollama(question)
        print(ollama_response)
    except Exception as e:
        print(f"Ollama call failed: {e}")

print("\n" + "="*60 + "\n")

In [ ]:
run_explainer()

---

## Audio Output Upgrade

The purpose of this tool is to make technical explanations work without visual output. Converting the text output to speech closes the feedback loop the tool was designed for: an explanation that sounds good when spoken is fundamentally different from one that merely avoids visual metaphors on the page, and the only way to verify that difference is to hear it.

The cells below add three functions — `generate_audio`, `explain_and_speak_with_anthropic`, and `explain_and_speak_with_ollama` — plus an updated entry point, `run_explainer_with_audio`, that generates spoken audio for both model responses inline in the notebook.

### `generate_audio`

Calls the OpenAI TTS API with `model="tts-1"` and the specified voice. Returns an IPython `Audio` widget that renders inline in the notebook cell output. Audio is not set to autoplay so the user controls playback — important for screen reader users who may not expect sound to start immediately.

Available voices: `alloy`, `echo`, `fable`, `onyx`, `nova`, `shimmer`. The default is `nova` because its pacing is clear and measured, which suits the numbered-step format of the explanations.



In [ ]:
openai = OpenAI()

def generate_audio(text: str, voice: str = "onyx") -> Audio:
    """Convert an explanation text string to a playable IPython Audio widget."""
    response = openai.audio.speech.create(
        model="tts-1", 
        voice=voice, 
        input=text
    )

    return Audio(data=response.content, rate=24000)

### `explain_and_speak_with_anthropic`

Wraps `explain_with_anthropic` and adds spoken output. After the text explanation is generated it passes the full text to `generate_audio` and renders the resulting `Audio` widget inline in the notebook for the user to play at their discretion — audio does not start automatically. Returns both the text and the audio object so the caller can display or save either independently.


In [ ]:
def explain_and_speak_with_anthropic(
    question: str,
    voice: str = "onyx",
) -> tuple[str, Audio]:
    """Explain a concept via Claude and generate spoken audio output."""
    text = explain_with_anthropic(question)
    audio = generate_audio(text, voice)
    display(audio)
    return (text, audio)


### `explain_and_speak_with_ollama`

Offline counterpart to `explain_and_speak_with_anthropic`. Wraps `explain_with_ollama`, converts the response to speech using `generate_audio`, and renders the `Audio` widget inline. Same return shape as the Anthropic version so the two are interchangeable in the run loop.



In [ ]:
def explain_and_speak_with_ollama(
    question: str,
    model: str = "llama3.2",
    voice: str = "nova",
) -> tuple[str, Audio]:
    """Explain a concept via Ollama and generate spoken audio output."""
    text = explain_with_ollama(question, model)
    audio = generate_audio(text, voice)
    display(audio)
    return (text, audio)

### `run_explainer_with_audio`

Replaces the original `run_explainer` loop for the audio upgrade. Takes the same user question, runs it through both models, and after printing each text response also renders the spoken audio inline via the Jupyter `Audio` widget for the user to play at their discretion — audio does not start automatically. A single optional `voice` parameter is forwarded to both speak functions so one call controls the voice for the entire session.



In [ ]:
import gradio as gr

def talker(text: str, voice: str = "onyx") -> bytes:
    response = openai.audio.speech.create(
        model="tts-1",
        voice=voice,
        input=text,
    )
    return response.content

def run_explainer_with_audio(voice: str = "onyx") -> None:
    """Gradio chat interface — explains concepts with audio from both models."""

    def chat(history):
        history = [{"role": h["role"], "content": h["content"]} for h in history]
        user_message = history[-1]["content"]

        anthropic_text = None
        try:
            anthropic_text = explain_with_anthropic(user_message)
        except Exception as e:
            anthropic_text = f"Anthropic call failed: {e}"

        ollama_text = None
        try:
            ollama_text = explain_with_ollama(user_message)
        except Exception as e:
            ollama_text = f"Ollama call failed: {e}"

        history += [
            {"role": "assistant", "content": f"[Claude]\n{anthropic_text}"},
            {"role": "assistant", "content": f"[Ollama / llama3.2]\n{ollama_text}"},
        ]

        audio = None
        try:
            audio = talker(anthropic_text, voice)
        except Exception as e:
            print(f"Audio generation failed: {e}")

        return history, audio

    def put_message_in_chatbot(message, history):
        return "", history + [{"role": "user", "content": message}]

    with gr.Blocks(title="Accessible Technical Concept Explainer") as demo:
        gr.Markdown("## Accessible Technical Concept Explainer")
        gr.Markdown(
            "Explanations are structured for screen reader and audio consumption. "
            "Type a technical question and press Enter or click Explain. "
            "Audio plays the Claude response."
        )
        chatbot = gr.Chatbot(label="Conversation History", height=500, type="messages")
        audio_out = gr.Audio(label="Spoken Explanation (Claude)", autoplay=False)
        with gr.Row():
            msg = gr.Textbox(
                label="Your technical question",
                placeholder="e.g. What is a closure in JavaScript?",
                scale=4,
            )
            submit_btn = gr.Button("Explain", scale=1)

        msg.submit(put_message_in_chatbot, [msg, chatbot], [msg, chatbot]).then(
            chat, inputs=chatbot, outputs=[chatbot, audio_out]
        )
        submit_btn.click(put_message_in_chatbot, [msg, chatbot], [msg, chatbot]).then(
            chat, inputs=chatbot, outputs=[chatbot, audio_out]
        )

    demo.launch()


In [ ]:
run_explainer_with_audio()